# Video Tokenizer Training (Spacetime Vector Quantized Variational Autoencoder)

In [1]:
import torch 

from torch.utils.data import DataLoader
from torchvision.datasets import UCF101

import lightning as L

# Import the model Arch from /models

In [2]:
from spacetime.models.tokenizers import STVQVae

## Lightning module

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [3]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        latent_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.25
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            latent_dim=latent_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout
        )
        self.beta = beta

    def forward(self, inputs):
        return self.model(inputs)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)
        
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)
        
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=0.01)

## Load UCF101 Action Recognition dataset 

We use the UCF101 dataset which contains 13,320 videos from 101 action categories. This dataset is commonly used for benchmarking video action recognition models, such as basketball shooting, biking, diving, golf swinging, horse riding, and playing musical instruments.


We created a subset of UCF101 with only 10 classes for faster experimentation. The selected classes are:
ApplyEyeMakeup, ApplyLipstick, Archery, BabyCrawling, BalanceBeam, BandMarching, BaseballPitch, Basketball, BasketballDunk, and BenchPress.

In [4]:
train_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,  # non overlapping clips
    train=True,
)

test_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,
    train=False,
)

print(f"Downsized train dataset size: {len(train_dataset)} clips")
print(f"Downsized test dataset size: {len(test_dataset)} clips")

  0%|          | 0/86 [00:00<?, ?it/s]/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
  0%|          | 0/86 [00:00<?, ?it/s]/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 86/86 [00:29<00:00,  2.90it/s]

Downsized train dataset size: 17561 clips
Downsized test dataset size: 6464 clips


In [5]:
import torch.nn.functional as F


def collate_ucf101(batch):
    # batch: list of (video, label, index) where label is detection labels 
    # and index is the index of the class for recognition
    xs, ys = [], []
    for v, _, l in batch:
        # v: T, H, W, C  (uint8)
        v = v.permute(0, 3, 1, 2)            # -> T, C, H, W
        v = v.float() / 255.0
        v = F.interpolate(v, size=(224, 224), mode='bilinear', align_corners=False)  # resize frames
        v = v.permute(1, 0, 2, 3).contiguous()  # -> C, F, H, W
        xs.append(v.clone())                  # new storage
        ys.append(int(l))
    return torch.stack(xs, 0), torch.tensor(ys, dtype=torch.long)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    collate_fn=collate_ucf101,
    pin_memory=True,
)

test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_ucf101)

In [6]:
lightning_timesformer = STVQVaeModule(
    num_heads=4,
    d_model=512,
    num_layers=4,
    d_linear=512,
    codebook_size=512,
    latent_dim=256,
    patch_size=16,
    frame_height=224,
    frame_width=224,
    num_frames=8,
    num_linear_layers=2,
    num_groups=8,
    dropout=0.1,
)

trainer = L.Trainer(max_epochs=1, precision=16, fast_dev_run=True)
trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader)

/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages

Training: |          | 0/? [00:00<?, ?it/s]

/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The vi